# 0 — Build pretrained model

Use this notebook only when you want to train a model from your own high-resolution tracking data in `data/raw/`. If you only want zero-shot use of the included pretrained model, skip this notebook.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)
print("PROJECT_ROOT:", PROJECT_ROOT, flush=True)

In [ ]:
from carnivore_reconstruction import DatasetSpec, ReconstructionConfig
from carnivore_reconstruction.timing import TimerLog, status
from carnivore_reconstruction.formal_large_felid import (
    filter_dataset_specs_large_felids,
    filter_large_felid_tracks,
    filter_large_felid_tasks,
    assign_formal_individual_split,
    write_analysis_decisions,
    formal_split_report,
    prepare_formal_task_tables,
)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "pretrained_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Formal split seed should stay fixed once the formal analysis starts.
FORMAL_SPLIT_SEED = 20260625

# Edit config/dataset_specs.py before running this notebook on your own data.
from config.dataset_specs import DATASETS as RAW_DATASETS

# For the original large-felid paper workflow, keep this filter.
# If training on a different taxon/study system, replace the next line with:
# DATASETS = RAW_DATASETS
DATASETS = filter_dataset_specs_large_felids(RAW_DATASETS)

config = ReconstructionConfig(
    output_dir=OUTPUT_DIR,
    task_sampling_mode="full_generalization",
    split_strategy="grouped",  # overwritten below by individual-disjoint formal split
    train_fraction=0.70,
    validation_fraction=0.15,
    max_tasks_per_animal_setting=None,
    max_animals_per_dataset_taxon_setting=None,
    max_tasks_per_dataset_taxon_setting=None,
    max_motifs=25000,
    retrieve_per_task=20,
    max_candidates_per_task=45,
    output_top_k=5,
    representative_top_k=20,
    save_candidate_paths=False,
    use_environmental_covariates=True,
    evaluate_environmental_exposure=False,
)
print("Configured datasets:", [d.dataset for d in DATASETS])


In [ ]:
from carnivore_reconstruction import load_datasets, make_tasks_from_tracks, MotifReconstructionModel
from carnivore_reconstruction.utils import write_table, write_json
from carnivore_reconstruction.transfer import (
    individual_transfer_units,
    task_counts_by_species_habitat,
    transfer_scenario_counts,
    add_transfer_support_labels,
)

timer = TimerLog()
write_analysis_decisions(OUTPUT_DIR, split_seed=FORMAL_SPLIT_SEED)

status("Build 1/6: loading and filtering high-resolution large-felid tracks")
with timer.step("load_large_felid_tracks"):
    tracks = load_datasets(DATASETS, verbose=True)
    tracks = filter_large_felid_tracks(tracks)
timer.save(OUTPUT_DIR / "runtime_build_model_partial.csv")

print("Tracks by taxon:")
display(tracks.groupby(["dataset", "taxon"])["animal_id"].nunique().reset_index(name="n_individuals"))

status("Build 2/6: creating candidate LR/HR reconstruction tasks")
with timer.step("make_reconstruction_tasks"):
    task_table, task_points, tasks = make_tasks_from_tracks(tracks, config)
    task_table, task_points = filter_large_felid_tasks(task_table, task_points)

status("Build 3/6: assigning formal individual-disjoint train/validation/test split")
with timer.step("assign_formal_individual_split", n_tasks=len(task_table)):
    task_table, formal_split_table = assign_formal_individual_split(
        task_table,
        stratify_cols=["taxon"],
        seed=FORMAL_SPLIT_SEED,
        train_fraction=config.train_fraction,
        validation_fraction=config.validation_fraction,
    )
    # Rebuild tasks after final split metadata and large-felid filtering.
    from carnivore_reconstruction import tasks_from_tables
    tasks = tasks_from_tables(task_table, task_points)

task_table, task_points, formal_split_table = prepare_formal_task_tables(
    task_table,
    task_points,
    formal_split_table,
    seed=FORMAL_SPLIT_SEED,
    train_fraction=config.train_fraction,
    validation_fraction=config.validation_fraction,
    label="formal_task_table",
)
# Rebuild tasks after any automatic cleanup/repair.
from carnivore_reconstruction import tasks_from_tables
tasks = tasks_from_tables(task_table, task_points)
formal_split_table.to_csv(OUTPUT_DIR / "formal_individual_split_table.csv", index=False)
write_json(formal_split_report(task_table, formal_split_table), OUTPUT_DIR / "formal_split_report.json")
write_table(task_table, OUTPUT_DIR / "task_table.parquet")
write_table(task_points, OUTPUT_DIR / "task_points.parquet")
timer.save(OUTPUT_DIR / "runtime_build_model_partial.csv")

print("Task table:", task_table.shape)
print("Tasks by split:", task_table["split"].value_counts().to_dict())
display(task_table.groupby(["dataset", "taxon", "setting_name", "split"]).size().reset_index(name="n_tasks"))
print("Individuals by split:")
display(formal_split_table.groupby(["taxon", "split"]).size().reset_index(name="n_individuals"))

In [ ]:
from carnivore_reconstruction.transfer import (
    individual_transfer_units,
    task_counts_by_species_habitat,
    transfer_scenario_counts,
    add_transfer_support_labels,
)

TRANSFER_DIR = OUTPUT_DIR / "transfer_labels"
TRANSFER_DIR.mkdir(parents=True, exist_ok=True)

status("Build 4/6: attaching transfer-ready species/habitat labels")
with timer.step("build_transfer_labels", n_tasks=len(task_table)):
    transfer_labels = add_transfer_support_labels(
        task_table,
        source_split="train",
        target_split=None,
        exclude_same_animal=True,
    )
    if not transfer_labels.empty:
        label_cols = [c for c in transfer_labels.columns if c != "task_uid"]
        task_table = task_table.drop(columns=[c for c in label_cols if c in task_table.columns], errors="ignore")
        task_table = task_table.merge(transfer_labels, on="task_uid", how="left")
        transfer_labels.to_csv(TRANSFER_DIR / "task_transfer_support_labels_train_to_all.csv", index=False)
        write_table(task_table, OUTPUT_DIR / "task_table.parquet")

    units = individual_transfer_units(task_table)
    task_counts = task_counts_by_species_habitat(task_table)
    train_to_validation = transfer_scenario_counts(task_table, source_split="train", target_split="validation", exclude_same_animal=True)
    train_to_test = transfer_scenario_counts(task_table, source_split="train", target_split="test", exclude_same_animal=True)

    units.to_csv(TRANSFER_DIR / "individual_transfer_units.csv", index=False)
    task_counts.to_csv(TRANSFER_DIR / "task_counts_by_species_habitat_split.csv", index=False)
    train_to_validation.to_csv(TRANSFER_DIR / "transfer_scenario_counts_train_to_validation.csv", index=False)
    train_to_test.to_csv(TRANSFER_DIR / "transfer_scenario_counts_train_to_test.csv", index=False)

display(train_to_test.head(30))
timer.save(OUTPUT_DIR / "runtime_build_model_partial.csv")

In [ ]:
status("Build 5/6: fitting formal large-felid model")
model = MotifReconstructionModel(config)
AUTO_N_JOBS = max(1, (os.cpu_count() or 2) - 1)
model.config.n_jobs = AUTO_N_JOBS
model.config.parallel_threshold_tasks = 2
print(f"Using n_jobs={AUTO_N_JOBS}", flush=True)

with timer.step("fit_pretrained_large_felid_model", n_tasks=len(tasks)):
    model.fit(tasks, task_table=task_table, train_split="train", timer=timer)

status("Build 6/6: saving model and metadata")
with timer.step("save_pretrained_model"):
    model_path = model.save(OUTPUT_DIR)
    timer.save(OUTPUT_DIR / "runtime_build_model.csv")

print("Saved model:", model_path)
print("Saved task table:", OUTPUT_DIR / "task_table.parquet")
print("Saved formal split table:", OUTPUT_DIR / "formal_individual_split_table.csv")